In [ ]:
from elasticsearch import Elasticsearch
import json
import numpy as np
from scipy.spatial.distance import cosine


In [114]:
"""
Comprehensive test suite for E5 embedding search
Tests semantic search quality with various queries
"""

 ============================================================================
# constants
# ============================================================================
env = "DEV"
MODEL_ID = "intfloat__e5-small"
MODEL_NAME = "e5"
SEARCH_TYPE = "vector"
#INDEX_RAW = "pp_vs_raw_docs-v1"        # stores original raw documents
INDEX_VECTOR = "pp-vs-e5-llmd-embeddings-no-passage"  # stores documents with vector embeddings
INDEX_NAME = "pp-vs-e5-llmd-embeddings-no-passage"
PIPELINE_ID = f"{MODEL_NAME.lower()}_embedding_base_pipeline"
SHOULD_DELETE_INDEX = False
#CLOUD_ID="dsg-search-dev-east:ZWFzdHVzLmF6dXJlLmVsYXN0aWMtY2xvdWQuY29tOjQ0MyQ1Zjc1NDk5MTVmNmE0ZTBjYWFhYzYzM2IwZDI1MjNlZSRiZTg1OWU2ZGM0ZTg0Y2RmYTVjNDNkNzJiZDA3ZTg3Mg=="
ES_HOST = os.getenv("ELASTIC_HOST_DEV")
ES_USER = os.getenv("ELASTIC_USERNAME")
ES_PASS = os.getenv("ELASTIC_PASSWORD")
INDEX_SETTINGS = { 
    "index": {
        "number_of_replicas": "0",
        "number_of_shards": "1",
        #"knn": True  # Enable kNN search
        # No need for "knn": true — only needed if using legacy knn_vector
    }   
}
#both sbert and e5 small has same dimension.
dims = 384 if MODEL_NAME.lower() == "sbert" else 384  # adjust per model


In [115]:
def get_embedding_column(model_name: str) -> str:
    return "vs_e5_llm_product_desc_embedding" if model_name.lower() == "e5" else "content_embedding"


In [116]:
# ============================================================================
# CONNECT TO ELASTICSEARCH
# ============================================================================

es = Elasticsearch(
    ES_HOST,  # Replace with your actual host
    basic_auth=(ES_USER,ES_PASS),
    request_timeout=600,
    verify_certs=True  # Optional: Set to False only if using self-signed certs
)

# Test connection
print(es.info().body)


{'name': 'instance-0000000096', 'cluster_name': '5f7549915f6a4e0caaac633b0d2523ee', 'cluster_uuid': 'hrds7YuPRLy60DbQMfN9Ow', 'version': {'number': '8.19.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '1fde05a4d63448377eceb8fd3d51ce16ca3f02a9', 'build_date': '2025-08-26T02:35:34.366492370Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [ ]:

# ============================================================================
# RUN QUERY
# ============================================================================

def run_query(
    es: Elasticsearch,
    query_text: str,
    model_id: str,
    index_name: str,
    size: int = 10
) -> list:
    """
    Run a KNN query using text_embedding (via query_vector_builder)
    """
    
    knn_query_body = {
        "size": size,
        "knn": {
            "field": "vs_e5_llm_product_desc_embedding",
            "k": size,
            "num_candidates": max(50 * size, 1000),
            "query_vector_builder": {
                "text_embedding": {
                    "model_id": model_id,
                    "model_text": query_text  # Plain text, NO prefix
                }
            }
        }
    }
    
    response = es.search(index=index_name, body=knn_query_body)
    
    results = [
        {
            "score": hit["_score"],
            "title": hit["_source"].get("llm_description", "N/A"),
            "id": hit.get("_id", "N/A")
        }
        for hit in response["hits"]["hits"]
    ]
    
    return results


# ============================================================================
# DISPLAY RESULTS
# ============================================================================

def display_results(query_text: str, results: list, top_k: int = 5) -> None:
    """
    Pretty print search results
    """
    
    print(f"\nQuery: '{query_text}'")
    print("=" * 100)
    print(f"{'Rank':<5} {'Score':<10} {'Product Title':<80}")
    print("-" * 100)
    
    for i, result in enumerate(results[:top_k], 1):
        score = result['score']
        title = result['title'][:75]
        print(f"{i:<5} {score:<10.6f} {title}...")
    
    print()


# ============================================================================
# EVALUATE RESULTS
# ============================================================================

def evaluate_results(query_text: str, results: list, expected_keywords: list) -> dict:
    """
    Evaluate if results are relevant
    
    Args:
        query_text: The search query
        results: List of search results
        expected_keywords: Keywords that should appear in relevant results
    
    Returns:
        Dictionary with evaluation metrics
    """
    
    relevant_count = 0
    irrelevant_count = 0
    partial_count = 0
    
    for i, result in enumerate(results[:5]):  # Check top 5
        title = result['title'].lower()
        
        # Check how many expected keywords match
        matches = sum(1 for keyword in expected_keywords if keyword.lower() in title)
        
        if matches == len(expected_keywords):
            relevant_count += 1
        elif matches > 0:
            partial_count += 1
        else:
            irrelevant_count += 1
    
    return {
        "relevant": relevant_count,
        "partial": partial_count,
        "irrelevant": irrelevant_count,
        "score": (relevant_count + 0.5 * partial_count) / 5  # Out of 5
    }


# ============================================================================
# TEST SUITE
# ============================================================================

def run_test_suite(es: Elasticsearch, index_name: str, model_id: str) -> None:
    """
    Run comprehensive tests on various queries
    """
    
    print("=" * 100)
    print("E5 EMBEDDING SEARCH TEST SUITE")
    print("=" * 100)
    
    # Define test queries with expected keywords
    test_cases = [
        {
            "query": "feather coat",
            "expected": ["feather", "coat"],
            "description": "Winter coat with feather insulation"
        },
        {
            "query": "down jacket",
            "expected": ["down", "jacket"],
            "description": "Winter jacket with down insulation"
        },
        {
            "query": "volleyball bag",
            "expected": ["volleyball", "bag"],
            "description": "Sports bag for volleyball equipment"
        },
        {
            "query": "running shoes",
            "expected": ["running", "shoes"],
            "description": "Athletic shoes for running"
        },
        {
            "query": "winter coat",
            "expected": ["winter", "coat"],
            "description": "Coat for cold weather"
        },
        {
            "query": "sports necklace",
            "expected": ["necklace"],
            "description": "Jewelry for sports fans"
        },
    ]
    
    results_summary = []
    
    for test_case in test_cases:
        query = test_case["query"]
        expected = test_case["expected"]
        description = test_case["description"]
        
        # Run query
        results = run_query(es, query, model_id, index_name, size=10)
        
        # Display results
        display_results(query, results, top_k=5)
        
        # Evaluate
        evaluation = evaluate_results(query, results, expected)
        
        # Store summary
        results_summary.append({
            "query": query,
            "description": description,
            "evaluation": evaluation,
            "top_result": results[0]["title"][:60] if results else "No results"
        })
        
        # Print evaluation
        print(f"Evaluation:")
        print(f"  ✓ Relevant (top 5): {evaluation['relevant']}")
        print(f"  ~ Partial match: {evaluation['partial']}")
        print(f"  ✗ Irrelevant: {evaluation['irrelevant']}")
        print(f"  Score: {evaluation['score']:.1%}")
        print()
    
    # Print summary
    print("\n" + "=" * 100)
    print("TEST SUMMARY")
    print("=" * 100)
    print(f"{'Query':<20} {'Description':<35} {'Relevant':<12} {'Score':<10}")
    print("-" * 100)
    
    total_score = 0
    for summary in results_summary:
        query = summary["query"][:18]
        desc = summary["description"][:33]
        relevant = f"{summary['evaluation']['relevant']}/5"
        score = f"{summary['evaluation']['score']:.1%}"
        
        print(f"{query:<20} {desc:<35} {relevant:<12} {score:<10}")
        total_score += summary['evaluation']['score']
    
    avg_score = total_score / len(results_summary)
    print("-" * 100)
    print(f"{'AVERAGE SCORE':<20} {'':<35} {'':<12} {avg_score:.1%}")
    print()
    
    # Interpretation
    print("\n" + "=" * 100)
    print("INTERPRETATION")
    print("=" * 100)
    
    if avg_score >= 0.8:
        print("✅ EXCELLENT! Your embeddings are working well.")
        print("   - Semantic search is capturing relevant products")
        print("   - Product types are properly separated")
    elif avg_score >= 0.6:
        print("✓ GOOD! Your embeddings are mostly working.")
        print("   - Some irrelevant results appearing")
        print("   - Consider adding metadata filtering")
    elif avg_score >= 0.4:
        print("⚠️  MODERATE. Results need improvement.")
        print("   - Check if documents have plain text (no 'passage:' prefix)")
        print("   - Verify pipeline is being used during indexing")
        print("   - Consider re-indexing with correct pipeline")
    else:
        print("❌ POOR. Something is wrong.")
        print("   - Check if pipeline is actually running")
        print("   - Verify documents are being indexed with the pipeline")
        print("   - Check if embeddings are being generated")


# ============================================================================
# SPECIFIC PROBLEM TEST (Your issue)
# ============================================================================

def test_feather_coat_specifics(es: Elasticsearch, index_name: str, model_id: str) -> None:
    """
    Specific test for the "feather coat" issue
    Check if neckties and fishing feathers are properly separated
    """
    
    print("\n" + "=" * 100)
    print("SPECIFIC TEST: Feather Coat Query")
    print("=" * 100)
    
    query = "feather coat"
    results = run_query(es, query, model_id, index_name, size=20)
    
    print(f"\nQuery: '{query}'")
    print("Checking if wrong products appear in top 10...\n")
    
    wrong_products = {
        "necklace": [],
        "necktie": [],
        "fishing_feathers": [],
        "sports_apparel": [],
        "correct_coat": []
    }
    
    for i, result in enumerate(results[:10], 1):
        title = result['title'].lower()
        score = result['score']
        
        # Categorize
        if "necklace" in title:
            category = "necklace"
            status = "❌ WRONG"
        elif "necktie" in title or "tie" in title:
            category = "necktie"
            status = "❌ WRONG"
        elif "feather" in title and ("fly" in title or "fishing" in title or "hackle" in title):
            category = "fishing_feathers"
            status = "❌ WRONG"
        elif any(sport in title for sport in ["nba", "nfl", "nhl", "mlb", "team", "sports"]):
            category = "sports_apparel"
            status = "⚠️  CAUTION"
        elif any(word in title for word in ["coat", "jacket", "parka"]) and any(word in title for word in ["feather", "down", "insulated", "warm"]):
            category = "correct_coat"
            status = "✅ CORRECT"
        else:
            category = None
            status = "?"
        
        print(f"{i}. Score: {score:.6f} {status}")
        print(f"   {result['title'][:70]}...\n")
    
    # Summary
    print("\n" + "=" * 100)
    print("RESULT ANALYSIS")
    print("=" * 100)
    
    # Count categories
    wrong_count = 0
    correct_count = 0
    
    for result in results[:10]:
        title = result['title'].lower()
        if any(word in title for word in ["coat", "jacket", "parka"]) and any(word in title for word in ["feather", "down", "insulated", "warm"]):
            correct_count += 1
        elif any(prod in title for prod in ["necklace", "necktie", "tie", "fly", "hackle", "fishing"]):
            wrong_count += 1
    
    print(f"Correct winter coats in top 10: {correct_count}")
    print(f"Wrong products in top 10: {wrong_count}")
    
    if correct_count >= 7:
        print("\n✅ EXCELLENT! Feather coat search is working correctly!")
    elif correct_count >= 5:
        print("\n✓ GOOD! Most results are correct.")
    elif correct_count >= 3:
        print("\n⚠️  MODERATE. Some irrelevant results.")
    else:
        print("\n❌ POOR. Too many wrong results.")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    
    # Configuration
    HOST = "http://localhost:9200"  # ← Change to your ES host
    USER = "elastic"                # ← Change to your username
    PASSWORD = "password"           # ← Change to your password
    
    INDEX_NAME = "pp-vs-e5-llmd-embeddings"
    MODEL_ID = "intfloat__e5-small"
    
    # Connect
    print("Connecting to Elasticsearch...")
    try:
        es = connect_elasticsearch(HOST, USER, PASSWORD)
        print("✅ Connected\n")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        exit(1)
    
    # Run tests
    run_test_suite(es, INDEX_NAME, MODEL_ID)
    
    # Run specific problem test
    test_feather_coat_specifics(es, INDEX_NAME, MODEL_ID)